# Mock Eikon Data API — Unstructured News Retrieval

This notebook reproduces the logic of the Eikon news example in an offline, mock form.

It covers:
- searching news headlines
- selecting one headline
- retrieving the full story by story ID
- rendering the story as HTML in Jupyter


In [1]:
import pandas as pd
import numpy as np
from IPython.display import HTML, display

pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 120)


In [4]:
class MockEikon:
    def __init__(self):
        self.app_key = None
        self._stories = self._build_stories()

    def set_app_key(self, key):
        self.app_key = key
        print(f"[MockEikon] App key set: {key}")

    def _build_stories(self):
        rows = [
            {
                'versionCreated': '2020-04-20 21:33:37.332000+00:00',
                'text': 'Apple said to launch new AirPods, MacBook Pro next month',
                'storyId': 'urn:newsml:reuters.com:20200420:nNRAble9rq:1',
                'sourceCode': 'NS:TIMIND',
                'body': (
                    '<p>NEW DELHI: Apple recently launched its much-awaited affordable smartphone iPhone SE. '
                    'Now it seems that the company is gearing up for another launch.</p>'
                    '<p>Apple is said to launch the next generation of AirPods and the all-new 13-inch MacBook Pro next month.</p>'
                    '<p>For Reprint Rights: timescontent.com</p>'
                )
            },
            {
                'versionCreated': '2020-04-20 10:20:23.201000+00:00',
                'text': 'Apple might launch upgraded AirPods, 13-inch MacBook Pro soon',
                'storyId': 'urn:newsml:reuters.com:20200420:nNRAbl8eob:1',
                'sourceCode': 'NS:BUSSTA',
                'body': '<p>Market chatter suggests Apple may refresh its AirPods lineup and MacBook Pro model.</p>'
            },
            {
                'versionCreated': '2020-04-20 02:32:27.721000+00:00',
                'text': 'Apple to reportedly launch new AirPods alongside MacBook Pro',
                'storyId': 'urn:newsml:reuters.com:20200420:nNRAbl4mfz:1',
                'sourceCode': 'NS:HINDUT',
                'body': '<p>Reports say Apple could unveil new AirPods together with the MacBook Pro.</p>'
            },
            {
                'versionCreated': '2020-04-15 12:06:58.693000+00:00',
                'text': 'Apple files a patent for iPhones, MacBook induction charging',
                'storyId': 'urn:newsml:reuters.com:20200415:nNRAbjvsix:1',
                'sourceCode': 'NS:HINDUT',
                'body': '<p>Apple has filed a patent related to charging devices such as iPhone and MacBook.</p>'
            },
            {
                'versionCreated': '2020-04-09 21:34:08.671000+00:00',
                'text': 'Apple rolls out new software update for MacBook users',
                'storyId': 'urn:newsml:reuters.com:20200409:nNRAbi2nbb:1',
                'sourceCode': 'NS:TIMIND',
                'body': '<p>Apple has released a software update aimed at improving the MacBook user experience.</p>'
            },
        ]
        df = pd.DataFrame(rows)
        df['versionCreated'] = pd.to_datetime(df['versionCreated'], utc=True)
        df = df.set_index('versionCreated').sort_index(ascending=False)
        return df

    def get_news_headlines(self, query, count=5, date_from=None, date_to=None):
        df = self._stories.copy()

        if date_from is not None:
            df = df[df.index >= pd.to_datetime(date_from, utc=True)]
        if date_to is not None:
            df = df[df.index <= pd.to_datetime(date_to, utc=True)]

        q = str(query).lower()
        terms = [t for t in q.replace(':', ' ').split() if t]
        for term in terms:
            if term == 'r' or term == 'aapl.o':
                continue
            df = df[df['text'].str.lower().str.contains(term, regex=False)]

        return df[['text', 'storyId', 'sourceCode']].head(count)

    def get_news_story(self, story_id):
        match = self._stories[self._stories['storyId'] == story_id]
        if match.empty:
            raise KeyError(f'Story not found: {story_id}')
        return f"<html><body>{match.iloc[0]['body']}</body></html>"


ek = MockEikon()
ek.set_app_key('DUMMY_KEY_123')


[MockEikon] App key set: DUMMY_KEY_123


## Retrieve news headlines

This follows the same logic as the book example: query Apple plus the word "Macbook", limit the result set to five items, and use a fixed date window.

In [5]:
headlines = ek.get_news_headlines(
    query='R:AAPL.O macbook',
    count=5,
    date_from='2020-4-1',
    date_to='2020-5-1'
)

headlines

,text,storyId,sourceCode
versionCreated,,,
2020-04-20 21:33:37.332000+00:00,"Apple said to launch new AirPods, MacBook Pro next month",urn:newsml:reuters.com:20200420:nNRAble9rq:1,NS:TIMIND
2020-04-20 10:20:23.201000+00:00,"Apple might launch upgraded AirPods, 13-inch MacBook Pro soon",urn:newsml:reuters.com:20200420:nNRAbl8eob:1,NS:BUSSTA
2020-04-20 02:32:27.721000+00:00,Apple to reportedly launch new AirPods alongside MacBook Pro,urn:newsml:reuters.com:20200420:nNRAbl4mfz:1,NS:HINDUT
2020-04-15 12:06:58.693000+00:00,"Apple files a patent for iPhones, MacBook induction charging",urn:newsml:reuters.com:20200415:nNRAbjvsix:1,NS:HINDUT
2020-04-09 21:34:08.671000+00:00,Apple rolls out new software update for MacBook users,urn:newsml:reuters.com:20200409:nNRAbi2nbb:1,NS:TIMIND


## Pick one story

The first row is selected as the story object.

In [6]:
story = headlines.iloc[0]
story

text          Apple said to launch new AirPods, MacBook Pro next month
storyId                   urn:newsml:reuters.com:20200420:nNRAble9rq:1
sourceCode                                                   NS:TIMIND
Name: 2020-04-20 21:33:37.332000+00:00, dtype: object

## Retrieve the full story body

The story ID is passed into the story retrieval function, which returns HTML.

In [7]:
news_text = ek.get_news_story(story['storyId'])
HTML(news_text)

## Notes

This notebook is a logical offline replica of the Eikon unstructured news workflow. The interface and output shape are intentionally similar to the book example, but the data is synthetic.